# QC participant assignments per rater

Summarizes the participants each rater is assigned in the rater config `RATER_CONFIG`
(QC-Studio's multi-user config format, see `ui/utils/user_config.py`):

1. **Config check** – valid vs rejected raters, missing participant lists.
2. **Per-rater counts** – rows, unique participants, sessions, duplicates, shared vs exclusive participants.
3. **Coverage across raters** – how many raters see each participant.
4. **Pairwise overlaps** – shared-participant counts and Jaccard index between every pair of raters.
5. **Overlap participants** – the participants rated by more than one rater (inter-rater pool).
6. **New rater list** – builds a participant list for a new rater from the least-covered participants.

Paths are resolved with the app's own `load_users_config`, so they match what each rater sees in QC-Studio.

In [ ]:
# --- Local paths --------------------------------------------------------------
# Machine-specific paths live in notebooks/local_config.json (gitignored).
# Create it by copying notebooks/local_config.example.json and editing the paths.
import json
from pathlib import Path

LOCAL_CONFIG = next(
    (p for p in (Path("local_config.json"), Path("notebooks") / "local_config.json") if p.is_file()), None
)
if LOCAL_CONFIG is None:
    raise FileNotFoundError(
        "notebooks/local_config.json not found: copy notebooks/local_config.example.json and set your paths"
    )
_local = json.loads(LOCAL_CONFIG.read_text())
print(f"Local config: {LOCAL_CONFIG.resolve()}")

RATER_CONFIG = _local["rater_config"]            # every rater, current and new
PARTICIPANT_LIST = _local["participant_list"]    # dataset-level participant pool (used by section 8)
OUTPUT_DIR = _local.get("output_dir")            # write summary TSVs here (None = don't save)

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Reuse the app's code so results match QC-Studio and its admin dashboard exactly.
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "ui" / "utils" / "user_config.py").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "ui"))
from utils.assignments import (
    create_rater_list,
    load_assignments,
    overlap_matrices,
    overlapping_pairs,
    rank_participants_by_rater_count,
    rater_count_distribution,
    rater_summary,
    assigned_raters_per_participant,
    read_participant_ids,
    restrict_to_pool,
    write_participant_list,
)
from utils.user_config import load_users_config

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

## 1. Load rater config

In [ ]:
cfg = load_users_config(RATER_CONFIG)
print(f"Config:     {cfg.path}")
print(f"Rater root: {cfg.rater_root}")
print(f"Valid raters: {len(cfg.users)}   Rejected: {len(cfg.rejected)}")

raters_df = pd.DataFrame(cfg.users.values())[
    ["rater_id", "email", "participant_list", "session_list", "out_dir"]
].sort_values("rater_id").reset_index(drop=True)
raters_df["list_exists"] = raters_df["participant_list"].map(lambda p: Path(p).is_file())
raters_df["list_file"] = raters_df["participant_list"].map(lambda p: Path(p).name)
display(raters_df[["rater_id", "email", "list_file", "list_exists", "session_list"]])

if cfg.rejected:
    print("\nRejected entries (these raters cannot sign in):")
    for email, reasons in cfg.rejected.items():
        print(f"  {email}: " + "; ".join(reasons))
missing = raters_df.loc[~raters_df["list_exists"], ["rater_id", "participant_list"]]
if not missing.empty:
    print("\nMissing participant lists (excluded below):")
    display(missing)

## 2. Load participant lists

In [ ]:
assign = load_assignments(cfg)
print(f"Assignments loaded: {len(assign)} rows across {assign['rater_id'].nunique()} raters")

## 3. Per-rater counts

In [ ]:
# How many raters each participant is assigned to
participant_raters = assigned_raters_per_participant(assign)
assign["n_raters_for_participant"] = assign["participant_id"].map(participant_raters)

per_rater = rater_summary(assign)
display(per_rater)
print(per_rater[["n_participants", "shared_with_others", "exclusive"]].describe().round(1))

## 4. Coverage across raters

How many distinct participants are assigned overall, and how many raters each one gets.

In [ ]:
print(f"Distinct participants across all raters: {len(participant_raters)}")
print(f"Sum of per-rater assignments: {int(per_rater['n_participants'].sum())}")

coverage = rater_count_distribution(assign)
display(coverage)

## 5. Pairwise overlaps between raters

Diagonal = each rater's participant count. Off-diagonal = participants shared by the pair.

In [ ]:
overlap, jaccard = overlap_matrices(assign)
raters = list(overlap.index)

print("Shared participant counts:")
display(overlap)

In [ ]:
# Pairs ranked by overlap (only pairs that share something)
pairs = overlapping_pairs(assign)
print(f"{len(pairs)} of {len(raters) * (len(raters) - 1) // 2} rater pairs share at least one participant")
if pairs["identical"].any():
    print("WARNING: some raters have identical lists:")
    display(pairs[pairs["identical"]])
display(pairs.head())

In [ ]:
# Optional heatmap (skipped if matplotlib isn't installed)
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib not installed; skipping heatmap")
else:
    fig, ax = plt.subplots(figsize=(0.5 * len(raters) + 3, 0.5 * len(raters) + 2))
    im = ax.imshow(jaccard.values, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(raters)), raters, rotation=90)
    ax.set_yticks(range(len(raters)), raters)
    for i in range(len(raters)):
        for j in range(len(raters)):
            ax.text(j, i, overlap.iat[i, j], ha="center", va="center", fontsize=7,
                    color="white" if jaccard.iat[i, j] > 0.5 else "black")
    fig.colorbar(im, ax=ax, label="Jaccard")
    ax.set_title("Rater overlap (cell text = shared count)")
    plt.tight_layout()
    plt.show()

## 6. Participants assigned to more than one rater

The inter-rater reliability pool.

In [ ]:
display(assign["n_raters_for_participant"].value_counts())

assign.head()

## 8. Create a participant list for a new rater

Builds the list for **one** new rater with a `POLICY`: `"least_rated"` takes the participants assigned to the **fewest raters**, `"random"` takes a seeded random sample. Participants are drawn from the dataset list, or from `CUSTOM_LIST` restricted to participants in the dataset list. The functions live in `ui/utils/assignments.py` (shared with the admin dashboard's Assignments tab).

* **Assignments come from the rater config** (`RATER_CONFIG`, which must list every rater, current and new): each rater's assignments are
  the `participant_list` it points to. A rater whose list doesn't exist yet (a new rater) has no assignments.
* `rank_participants_by_rater_count(rater_config, pool_ids)` gives every participant in the pool its number of raters
  sorted fewest first, with ties shuffled using `SEED`.
* `create_rater_list(rater_id, n, pool_ids, assignments, policy=POLICY)` takes `n` participants by the policy. Any existing
  rows for `rater_id` are ignored first, so re-running for the same rater gives the same list.
* `write_participant_list` writes to the `participant_list` path that the rater config gives that rater, only when
  `WRITE = True`, and never replaces an existing file unless `OVERWRITE = True`. The config then needs no edits.
* For several new raters, run this once per rater and write each list before the next run, so the next rater moves on
  to the next-least-covered participants.

In [ ]:
# --- Settings for the new list -------------------------------------------------
# RATER_CONFIG (every rater, current and new) and PARTICIPANT_LIST (global pool) come from local_config.json
NEW_RATER = "nikhil"                           # rater_id in RATER_CONFIG
N_PARTICIPANTS = 100                             # how many participants to assign
POLICY = "least_rated"                           # "least_rated" or "random" (see utils.assignments.POLICIES)
CUSTOM_LIST = None                               # None = whole dataset list; or a TSV path to draw only from those participants
SEED = 42
WRITE = False
OVERWRITE = False

In [ ]:
pool_ids = read_participant_ids(PARTICIPANT_LIST)
source_ids = pool_ids
if CUSTOM_LIST:
    source_ids, not_in_pool = restrict_to_pool(read_participant_ids(CUSTOM_LIST), pool_ids)
    print(f"Custom list: {len(source_ids)} participants in the dataset list, {len(not_in_pool)} left out {not_in_pool[:10]}")
ranked = rank_participants_by_rater_count(RATER_CONFIG, source_ids, seed=SEED, exclude_raters=[NEW_RATER])
print(f"Pool: {len(source_ids)} participants")
print("Raters per participant (before):")
display(ranked["n_raters"].value_counts().sort_index().rename_axis("n_raters").rename("n_participants").to_frame())
display(ranked.head(10))

In [ ]:
assignments = load_assignments(RATER_CONFIG, warn=lambda _: None)
new_list = create_rater_list(NEW_RATER, N_PARTICIPANTS, source_ids, assignments, seed=SEED, policy=POLICY,
                             warn_outside_pool=not CUSTOM_LIST)
print(f"{NEW_RATER}: {len(new_list)} participants; raters they had before:")
display(new_list["n_raters"].value_counts().sort_index().rename_axis("prior_n_raters").rename("n_participants").to_frame())
display(new_list.head())

if WRITE:
    out, status = write_participant_list(new_list["participant_id"], RATER_CONFIG, NEW_RATER, overwrite=OVERWRITE)
    print(f"{out}: {status}")
else:
    print("Dry run: set WRITE = True to save the list")